In [1]:
import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import os

# Create data directory
os.makedirs('data', exist_ok=True)
print("Generating the 4 Kaggle-style raw datasets...")

np.random.seed(42)
random.seed(42)
start_date = datetime(2024, 1, 1)

# --- DATASET 1: PATIENT ADMISSIONS (For Hospital Overview & Patient Flow) ---
admissions_data = []
depts = ['Cardio', 'Ortho', 'Ped', 'Gen Med', 'ICU', 'surgery', 'Cardiology', 'Pediatrics'] # intentionally messy
admission_types = ['Emergency', 'Outpatient', 'Inpatient', 'Day Care']
regions = ['North', 'South', 'East', 'West']

for i in range(1, 2501):
    ad_date = start_date + timedelta(days=random.randint(0, 364))
    stay_days = random.choices([random.randint(1, 4), random.randint(5, 10), random.randint(11, 20)], weights=[60, 30, 10])[0]
    dis_date = ad_date + timedelta(days=stay_days)

    # Introduce random missing values (NaN) to simulate messy raw data
    age = random.choice([random.randint(1, 90), np.nan])
    gender = random.choices(['Male', 'Female', np.nan], weights=[48, 48, 4])[0]

    admissions_data.append({
        'patient_id': f"PT{i:04d}",
        'hospital_id': random.choice([101, 102, 103, 104, 105]),
        'age': age,
        'gender': gender,
        'department': random.choice(depts),
        'admission_type': random.choice(admission_types),
        'admission_date': ad_date.strftime('%Y-%m-%d'),
        'discharge_date': dis_date.strftime('%Y-%m-%d'),
        'readmission_bite': random.choices([1, 0], weights=[8, 92])[0], # intentionally bad column name to fix later
        'region': random.choice(regions)
    })

df_admissions = pd.DataFrame(admissions_data)
# Add some duplicate rows to challenge our cleaning module
df_admissions = pd.concat([df_admissions, df_admissions.sample(50)]).reset_index(drop=True)
df_admissions.to_csv('data/raw_admissions.csv', index=False)

# --- DATASET 2: HOSPITAL BEDS CAPACITY (For Resource Utilization) ---
hospitals = [
    {'hospital_id': 101, 'hospital_name': 'City Care Hospital', 'total_beds': 500, 'icu_beds': 50},
    {'hospital_id': 102, 'hospital_name': 'Green Valley Hospital', 'total_beds': 350, 'icu_beds': 30},
    {'hospital_id': 103, 'hospital_name': 'Summit Medical Center', 'total_beds': 600, 'icu_beds': 60},
    {'hospital_id': 104, 'hospital_name': 'Metro Health Institute', 'total_beds': 450, 'icu_beds': 40},
    {'hospital_id': 105, 'hospital_name': 'HealthFirst Hospital', 'total_beds': 300, 'icu_beds': 25}
]
df_beds = pd.DataFrame(hospitals)
df_beds.to_csv('data/raw_hospital_beds.csv', index=False)

# --- DATASET 3: STAFF ALLOCATION (For Department Analytics) ---
staff_data = []
staff_roles = ['Doctor', 'Nurse', 'Technician', 'Administrative']
for h_id in [101, 102, 103, 104, 105]:
    for dept in ['Cardiology', 'Orthopedics', 'Pediatrics', 'General Medicine', 'ICU']:
        staff_data.append({
            'hospital_id': h_id,
            'department_clean': dept,
            'staff_count': random.randint(15, 80),
            'shift_type': 'Rotational'
        })
df_staff = pd.DataFrame(staff_data)
df_staff.to_csv('data/raw_staff_allocation.csv', index=False)

# --- DATASET 4: HEALTHCARE RESOURCES & EQUIPMENT (For Resource Utilization) ---
equipment_data = [
    {'hospital_id': 101, 'ventilators_total': 25, 'mri_machines': 3, 'ct_scanners': 4},
    {'hospital_id': 102, 'ventilators_total': 15, 'mri_machines': 2, 'ct_scanners': 2},
    {'hospital_id': 103, 'ventilators_total': 35, 'mri_machines': 5, 'ct_scanners': 6},
    {'hospital_id': 104, 'ventilators_total': 20, 'mri_machines': 3, 'ct_scanners': 3},
    {'hospital_id': 105, 'ventilators_total': 12, 'mri_machines': 1, 'ct_scanners': 2}
]
df_equipment = pd.DataFrame(equipment_data)
df_equipment.to_csv('data/raw_equipment_utilization.csv', index=False)

print("All 4 raw datasets successfully saved in the '/data' folder inside Colab!")

Generating the 4 Kaggle-style raw datasets...
All 4 raw datasets successfully saved in the '/data' folder inside Colab!


In [2]:
import pandas as pd
import numpy as np

print("Running Cleaning Pipeline...")

# Load the raw files
df_adm = pd.read_csv('data/raw_admissions.csv')
df_beds = pd.read_csv('data/raw_hospital_beds.csv')
df_staff = pd.read_csv('data/raw_staff_allocation.csv')
df_equip = pd.read_csv('data/raw_equipment_utilization.csv')

# 1. Deduplicate
df_adm = df_adm.drop_duplicates()

# 2. Handle missing patient values
df_adm['age'] = df_adm['age'].fillna(df_adm['age'].median())
df_adm['gender'] = df_adm['gender'].fillna('Not Disclosed')

# 3. Standardize and Fix Column Names / Department Names
df_adm = df_adm.rename(columns={'readmission_bite': 'readmission_rate'})
dept_mapping = {
    'Cardio': 'Cardiology', 'Cardiology': 'Cardiology',
    'Ortho': 'Orthopedics',
    'Ped': 'Pediatrics', 'Pediatrics': 'Pediatrics',
    'Gen Med': 'General Medicine', 'ICU': 'ICU', 'surgery': 'Surgery'
}
df_adm['department'] = df_adm['department'].map(dept_mapping).fillna('General Medicine')

# 4. Correct Date formatting
df_adm['admission_date'] = pd.to_datetime(df_adm['admission_date'])
df_adm['discharge_date'] = pd.to_datetime(df_adm['discharge_date'])
df_adm['length_of_stay'] = (df_adm['discharge_date'] - df_adm['admission_date']).dt.days

# 5. Merge all datasets seamlessly using relational links
# First, attach hospital bed capacity profiles
master_df = pd.merge(df_adm, df_beds, on='hospital_id', how='left')

# Next, attach machinery asset footprints
master_df = pd.merge(master_df, df_equip, on='hospital_id', how='left')

# Save the unified cleaned output file
master_df.to_csv('data/hospital_cleaned.csv', index=False)
print(f"Success! Integrated master file created with shape: {master_df.shape}")
print("You can now download 'hospital_cleaned.csv' from your files tab!")

Running Cleaning Pipeline...
Success! Integrated master file created with shape: (2500, 17)
You can now download 'hospital_cleaned.csv' from your files tab!


In [3]:
import pandas as pd
import numpy as np

print("Loading cleaned data for KPI Engineering...")
# Make sure your cleaned file is still in the Colab 'data' folder
df = pd.read_csv('data/hospital_cleaned.csv')

# --- Testing Core KPIs ---
total_admissions = len(df)
avg_length_of_stay = df['length_of_stay'].mean()
readmission_rate = (df['readmission_rate'].sum() / total_admissions) * 100
avg_occupancy = (total_admissions / df['total_beds'].sum()) * 100 # Simplified proxy for testing

print("--- HEALTHCARE KPIs VALIDATED ---")
print(f"Total Admissions: {total_admissions}")
print(f"Average Length of Stay: {avg_length_of_stay:.1f} days")
print(f"Overall Readmission Rate: {readmission_rate:.1f}%")
print(f"Estimated Bed Occupancy Rate: {avg_occupancy:.1f}%")

# --- Final Deliverable Export ---
# The project requires the final file to be an Excel workbook
output_file = 'data/hospital_final_dataset.xlsx'
df.to_excel(output_file, index=False)

print(f"\nSuccess! Final dataset saved as '{output_file}'.")
print("This is the exact file you will connect to Tableau!")

Loading cleaned data for KPI Engineering...
--- HEALTHCARE KPIs VALIDATED ---
Total Admissions: 2500
Average Length of Stay: 5.5 days
Overall Readmission Rate: 8.0%
Estimated Bed Occupancy Rate: 0.2%

Success! Final dataset saved as 'data/hospital_final_dataset.xlsx'.
This is the exact file you will connect to Tableau!
